# Binary-lens image-plane geometry

This notebook uses `lcbinint.image` to compute point-source image positions and make a clean geometry plot. It keeps the source-plane caustics and image-plane critical curves in separate panels so the coordinate roles are clear.

In [ ]:
from pathlib import Path
import sys

for build_dir in ("build", "build_new"):
    build_path = next(
        (root / build_dir
         for root in (Path.cwd(), *Path.cwd().parents)
         if (root / build_dir).is_dir()),
        None,
    )
    if build_path is not None:
        sys.path.insert(0, str(build_path))
        break

import matplotlib.pyplot as plt
import numpy as np

import lcbinint

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.18,
})

Choose a resonant planetary case and a source near the caustic. `q`, `s`, `x`, and `y` are the only required parameters.

In [ ]:
plane = lcbinint.image.binary(
    q=1.0e-3,
    s=1.0,
    x=0.01,
    y=-0.02,
    rho=2.0e-3,
    n_points=900,
)

images = plane.image_table()
[
    {
        "x": float(row["x"]),
        "y": float(row["y"]),
        "magnification": float(row["magnification"]),
        "parity": int(row["parity"]),
    }
    for row in images
]

The left panel is the source plane: caustics plus the source position and optional finite-source disk. The right panel is the image plane: critical curves plus the point-source images, colored by parity.

In [ ]:
def plot_branches(ax, branches, *, color, lw, label, alpha=1.0):
    for i, (xs, ys) in enumerate(zip(branches.x, branches.y)):
        ax.plot(xs, ys, color=color, lw=lw, alpha=alpha, label=label if i == 0 else None)


def set_square_limits(ax, xs, ys, *, pad=0.12):
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    cx = 0.5 * (np.nanmin(xs) + np.nanmax(xs))
    cy = 0.5 * (np.nanmin(ys) + np.nanmax(ys))
    span = max(np.nanmax(xs) - np.nanmin(xs), np.nanmax(ys) - np.nanmin(ys))
    span = span * (1.0 + pad) if span > 0 else 1.0
    ax.set_xlim(cx - 0.5 * span, cx + 0.5 * span)
    ax.set_ylim(cy - 0.5 * span, cy + 0.5 * span)


caustics = plane.caustics()
critical = plane.critical_curves()

fig, (ax_source, ax_image) = plt.subplots(1, 2, figsize=(11, 5.2), constrained_layout=True)

plot_branches(ax_source, caustics, color="#d43f3a", lw=1.5, label="caustic")
ax_source.scatter([plane.x], [plane.y], s=72, marker="*", color="#1f77b4", zorder=5, label="source")
ax_source.add_patch(plt.Circle((plane.x, plane.y), plane.rho, fill=False, color="#1f77b4", lw=1.1, alpha=0.8))
ax_source.set_title("Source plane")
ax_source.set_xlabel("source x")
ax_source.set_ylabel("source y")

source_xs = [x for branch in caustics.x for x in branch] + [plane.x - plane.rho, plane.x + plane.rho]
source_ys = [y for branch in caustics.y for y in branch] + [plane.y - plane.rho, plane.y + plane.rho]
set_square_limits(ax_source, source_xs, source_ys, pad=0.22)

plot_branches(ax_image, critical, color="0.35", lw=1.3, label="critical curve")
positive = images[images["parity"] > 0]
negative = images[images["parity"] < 0]
if len(positive):
    ax_image.scatter(positive["x"], positive["y"], s=62, color="#2ca25f", edgecolor="white", lw=0.7, zorder=5, label="image +")
if len(negative):
    ax_image.scatter(negative["x"], negative["y"], s=62, color="#5e3c99", edgecolor="white", lw=0.7, zorder=5, label="image -")
for i, row in enumerate(images):
    ax_image.annotate(str(i), (row["x"], row["y"]), xytext=(5, 5), textcoords="offset points", fontsize=9)
ax_image.set_title("Image plane")
ax_image.set_xlabel("image x")
ax_image.set_ylabel("image y")

image_xs = [x for branch in critical.x for x in branch] + images["x"].tolist()
image_ys = [y for branch in critical.y for y in branch] + images["y"].tolist()
set_square_limits(ax_image, image_xs, image_ys, pad=0.12)

for ax in (ax_source, ax_image):
    ax.set_aspect("equal", adjustable="box")
    ax.legend(frameon=False, loc="best")

fig.suptitle(f"Binary lens geometry: q={plane.q:g}, s={plane.s:g}", y=1.02, fontsize=14)
fig

For a compact one-panel view, use the convenience method directly:

In [ ]:
ax = plane.plot()
ax.set_title("Combined geometry view")
ax.figure